In [1]:
import mlflow, mlflow.pyfunc
from mlflow.models.signature import infer_signature
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np, os
mlflow.set_experiment("LinearRegression-Diabetes-Local")
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
with mlflow.start_run(run_name="linear-regression-diabetes-local") as run:
    m = LinearRegression().fit(X_train, y_train)
    p = m.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, p)))
    mae = float(mean_absolute_error(y_test, p))
    r2 = float(r2_score(y_test, p))
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    s = infer_signature(X_train, m.predict(X_train))
    mlflow.sklearn.log_model(m, "model", signature=s)
    run_id = run.info.run_id
    os.environ["RUN_ID"] = run_id
    print(run_id)
    print(f"runs:/{run_id}/model")

2025/11/03 00:02:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


25d50a31cda2465c80392f355b4572c8
runs:/25d50a31cda2465c80392f355b4572c8/model


In [2]:
reloaded = mlflow.pyfunc.load_model(f"runs:/{os.environ['RUN_ID']}/model")
print(reloaded.predict(X_test[:5]))

/opt/anaconda3/envs/mlopslab/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

[137.94908878 182.533354   129.85295373 292.5630923  124.86788221]


In [3]:
import subprocess, sys
ui = subprocess.Popen([sys.executable, "-m", "mlflow", "ui", "--port", "5000"]) 
print("http://127.0.0.1:5000")
print(ui.pid)

http://127.0.0.1:5000
62771


[2025-11-03 00:03:01 -0500] [62773] [INFO] Starting gunicorn 23.0.0
[2025-11-03 00:03:01 -0500] [62773] [INFO] Listening at: http://127.0.0.1:5000 (62773)
[2025-11-03 00:03:01 -0500] [62773] [INFO] Using worker: sync
[2025-11-03 00:03:01 -0500] [62774] [INFO] Booting worker with pid: 62774
[2025-11-03 00:03:01 -0500] [62775] [INFO] Booting worker with pid: 62775
[2025-11-03 00:03:01 -0500] [62776] [INFO] Booting worker with pid: 62776
[2025-11-03 00:03:01 -0500] [62784] [INFO] Booting worker with pid: 62784


In [4]:
import subprocess, sys, os
srv = subprocess.Popen([sys.executable, "-m", "mlflow", "models", "serve", "-m", f"runs:/{os.environ['RUN_ID']}/model", "-p", "5001", "-h", "0.0.0.0"]) 
print("http://127.0.0.1:5001/invocations")
print(srv.pid)

http://127.0.0.1:5001/invocations
62833


2025/11/03 00:03:12 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
Traceback (most recent call last):
  File "/opt/anaconda3/envs/mlopslab/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/envs/mlopslab/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/opt/anaconda3/envs/mlopslab/lib/python3.9/site-packages/mlflow/__main__.py", line 3, in <module>
    cli.main()
  File "/opt/anaconda3/envs/mlopslab/lib/python3.9/site-packages/click/core.py", line 1082, in main
    rv = self.invoke(ctx)
  File "/opt/anaconda3/envs/mlopslab/lib/python3.9/site-packages/click/core.py", line 1697, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
  File "/opt/anaconda3/envs/mlopslab/lib/python3.9/site-packages/click/core.py", line 1697, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
  File "/opt/anaconda3/envs/ml

In [5]:
import requests, json
cols = [f"x{i}" for i in range(10)]
payload = {"columns": cols, "instances": [X_test[0].tolist()]}
r = requests.post("http://127.0.0.1:5000/invocations", headers={"Content-Type": "application/json"}, data=json.dumps(payload))
print(r.status_code)
print(r.text)

404
<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>

